# โปรแกรมวิเคราะห์สัดส่วนยอดขายเกมแยกตามค่าย (Publisher)
**วิธีใช้งาน:** 
1. แก้ไขรายชื่อค่ายเกมที่ต้องการดูในกล่องโค้ดด้านล่าง (ตรงตัวแปร `target_publishers`)
2. กด **Run All** เพื่อประมวลผลตาราง สรุปข้อมูล และสร้างไฟล์กราฟ HTML อัตโนมัติ

In [ ]:
# =========================================================================
# 📝 [ส่วนสำหรับแก้ไขข้อมูล] เปลี่ยนชื่อค่ายเกมที่ต้องการตรงนี้ได้เลยครับ
# =========================================================================
target_publishers = [
    'Nintendo', 
    'Electronic Arts', 
    'Sony Computer Entertainment', 
    'Ubisoft', 
    'Take-Two Interactive'
]

In [ ]:
import pandas as pd
import plotly.express as px
import os

# 1. เรียกอ่านข้อมูลจากไฟล์
file_path = "../data/processed/Game_data_clean.xlsx"

if not os.path.exists(file_path):
    print(f"❌ ไม่พบไฟล์ข้อมูลในตำแหน่ง: {file_path} กรุณาตรวจสอบตำแหน่งไฟล์ใหม่อีกครั้ง")
else:
    df = pd.read_excel(file_path)
    
    # กรองข้อมูลเฉพาะค่ายเกมที่เลือกไว้ด้านบน
    df_filtered = df[df['Publisher'].isin(target_publishers)]

    if df_filtered.empty:
        print(f"⚠️ ไม่พบข้อมูลของค่ายเกมที่ระบุในระบบ กรุณาตรวจสอบตัวสะกดใหม่อีกครั้ง")
    else:
        # 2. เลือกเฉพาะ 3 ภูมิภาคหลัก แยกตามค่ายเกม
        main_regions = ["NA_Sales", "EU_Sales", "JP_Sales"]
        df_grouped = df_filtered.groupby("Publisher")[main_regions].sum()

        # 3. แปลงยอดขายให้เป็น "สัดส่วนเปอร์เซ็นต์ (%)"
        df_percentage = df_grouped.div(df_grouped.sum(axis=1), axis=0) * 100
        df_percentage.columns = ["NA (%)", "EU (%)", "JP (%)"]

        # 4. คำนวณค่าเฉลี่ยยอดขายรวมต่อเกม
        df_mean = df_filtered.groupby("Publisher")["Global_Sales"].mean().to_frame(name="Mean Sales")

        # 5. แสดงผลลัพธ์เป็นตาราง
        print("--- ตารางสัดส่วนยอดขาย (%) แยกตาม 3 ภูมิภาคหลัก ---")
        print(df_percentage.round(2))
        print("\n--- ค่าเฉลี่ยยอดขายรวมต่อเกม (ล้านชุด) ---")
        print(df_mean.round(2))

        # 6. แปลงข้อมูลทำ Heatmap ด้วย Plotly
        df_heatmap = df_percentage.reset_index().melt(
            id_vars='Publisher', 
            value_vars=["NA (%)", "EU (%)", "JP (%)"],
            var_name='Region', 
            value_name='Percentage'
        )

        # 7. สร้าง Interactive Heatmap
        fig = px.density_heatmap(
            df_heatmap,
            x="Region",
            y="Publisher",
            z="Percentage",
            text_auto=".1f",
            color_continuous_scale="YlOrRd",
            title="<b>Player Behavior: Sales Proportion (%) by Main Regions per Publisher</b>",
            labels={'Percentage': 'Percentage (%)'}
        )

        fig.update_layout(
            width=850,
            height=500,
            title_font_size=16,
            xaxis_title="Region",
            yaxis_title="Publisher",
            template="plotly_white"
        )

        fig.show()

        # 8. บันทึกไฟล์เป็น HTML
        output_filename = "publisher_sales_report.html"
        fig.write_html(output_filename)
        print(f"\n✨ [สำเร็จ] อัปเดตและบันทึกไฟล์กราฟส่งงานแล้วที่: {output_filename}")